In [1]:
from statsmodels.stats.multitest import multipletests

import pandas as pd
from scipy import stats
import numpy as np
import anndata as ad

adata = ad.read_h5ad("/mnt/lustre/scratch/nlsas/home/ulc/co/mao/modelo_prueba_final_fast_convergence/adata/mofa_adata_30f.h5ad")
factors_matrix = adata.X
num_factors = factors_matrix.shape[1]

for metadata_col in ['plate', 'drug', 'moa-fine', 'concentration']:
    results = []
    print(f"\nAnalizando contra '{metadata_col}'")

    for i in range(num_factors):
        df_temp = pd.DataFrame({
            'value': factors_matrix[:, i],
            'group': adata.obs[metadata_col].values
        }).dropna()

        groups = [g['value'].values for _, g in df_temp.groupby('group', observed=True)]
        if len(groups) > 1:
            f_stat, p_val = stats.f_oneway(*groups)
        else:
            f_stat, p_val = 0.0, 1.0

        results.append({
            'Factor': f"Factor{i+1}",
            'F_Stat': round(f_stat, 4),
            'P_Value': p_val,
        })

    df_res = pd.DataFrame(results)
    
    # Corrección FDR
    df_res['P_adj'] = multipletests(df_res['P_Value'], method='fdr_bh')[1]
    df_res['Veredicto'] = df_res['P_adj'].apply(
        lambda x: 'Significativo' if x < 0.05 else 'No significativo'
    )

    print(df_res.to_string(index=False))
    
    out = f'/home/ulc/co/mao/drug_clustering/TFM_mao/resultados/technical_batch/anova_{metadata_col.replace("-","_")}.xlsx'
    df_res.to_excel(out, index=False)
    print(f"Guardado en {out}")


Analizando contra 'plate'
  Factor  F_Stat  P_Value  P_adj        Veredicto
 Factor1  0.3779 0.976833    1.0 No significativo
 Factor2  0.0978 0.999983    1.0 No significativo
 Factor3  0.0518 1.000000    1.0 No significativo
 Factor4  0.0591 0.999999    1.0 No significativo
 Factor5  0.4034 0.969118    1.0 No significativo
 Factor6  0.1373 0.999879    1.0 No significativo
 Factor7  0.2903 0.993273    1.0 No significativo
 Factor8  0.0562 0.999999    1.0 No significativo
 Factor9  0.0282 1.000000    1.0 No significativo
Factor10  0.0336 1.000000    1.0 No significativo
Factor11  0.2101 0.998721    1.0 No significativo
Factor12  0.1832 0.999390    1.0 No significativo
Factor13  0.1416 0.999856    1.0 No significativo
Factor14  0.1336 0.999897    1.0 No significativo
Factor15  0.1111 0.999965    1.0 No significativo
Factor16  0.5039 0.923289    1.0 No significativo
Factor17  0.3286 0.987797    1.0 No significativo
Factor18  0.1114 0.999964    1.0 No significativo
Factor19  0.1168 0.9999